# Getting Started with harness in AgentCore 

Welcome to **harness in AgentCore** — it helps developers experiment and ship agents faster by letting them define and run agents in one API call, skipping framework setup, orchestration code, and deployment.

| Info | Details |
|---|---|
| Tutorial | Getting Started — Create, Invoke, ExecuteCommand |
| SDK | boto3 |
| Model | Claude Haiku 4.5 and Claude Sonnet 4.6 (Bedrock) |

This notebook walks you through:
1. Creating an IAM execution role
2. Creating a Harness agent
3. Invoking the agent with a prompt
4. Running commands on the agent's VM
5. Cleanup

### 1. Install Dependencies

In [1]:
!uv pip install -qU -r ../requirements.txt

error: No virtual environment found; run `uv venv` to create an environment, or pass `--system` to install into a non-virtual environment


In [ ]:
!uv pip freeze | grep boto3

Please restart your jupyter kernel before continuing.

### 2. Setup

Import the shared helper from the project root. These handle IAM role creation, boto3 client configuration, and lifecycle operations.

In [3]:
import sys
import time
import uuid
from pathlib import Path
import boto3

# helpers
sys.path.insert(0, str(Path.cwd().parent))

# --- Configuration ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- Create boto3 clients ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

Account: 016336666133


### 3. Create IAM Role

Create the IAM execution role with all necessary permissions for it. This is idempotent — if the role already exists, it returns the existing ARN.

In [4]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

Created role: arn:aws:iam::016336666133:role/HarnessExecutionRole
Attached policy: HarnessExecutionPolicy

Execution Role ARN: arn:aws:iam::016336666133:role/HarnessExecutionRole
Waiting for IAM role to propagate...
Ready!


### 4. Create Harness Agent

Create a new Harness agent with a unique name. The agent starts in `CREATING` status and transitions to `READY` after a few seconds.

In [6]:
HARNESS_NAME = f"GettingStarted_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=role_arn,
)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")
print(f"Harness ARN: {harness_arn}")
print(f"Status: {harness['status']}")

# Check for readiness
for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("\u2705 Harness is ready")
        break
    time.sleep(5)

Harness ID: GettingStarted_c94323ab-2mJIjUS4EN
Harness ARN: arn:aws:bedrock-agentcore:ap-south-1:016336666133:harness/GettingStarted_c94323ab-2mJIjUS4EN
Status: CREATING
Attempt 1: CREATING
Attempt 2: CREATING
Attempt 3: CREATING
Attempt 4: READY
✅ Harness is ready


### 5. Invoke Agent

Send a message to the agent. Each invocation needs a `session_id` that identifies the isolated microVM. The response is streamed — text and tool calls arrive as events.

> 💡 Change the message or model to experiment. The agent has `file_operations` and `shell` tools available by default.

In [10]:
session_id = str(uuid.uuid4()).upper()
print(f"Session ID: {session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "What are three fun things to do in Seattle on a rainy day?"
                    "Save your answer to a Markdown file."
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "apac.amazon.nova-lite-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

Session ID: 1F7EFC37-77BE-48FE-B42B-D48B08946F67

<thinking>The user is asking for a list of fun activities to do in Seattle on a rainy day. I can create a Markdown file with this information, but I don't have the exact details. I can use the shell tool to search for information online.</thinking>

[Tool: shell]


<thinking>It seems that the shell tool is not providing the required information due to JavaScript being blocked. I can't directly search for information online. However, I do have some knowledge about Seattle attractions. I can list a few fun things to do in Seattle on a rainy day and then save this information to a Markdown file.</thinking>

Here are three fun things to do in Seattle on a rainy day:

1. **Visit the Museum of Pop Culture (MoPOP)**: This museum is dedicated to contemporary popular culture, including music, movies, and video games. It's a great place to spend a rainy day.

2. **Explore Pike Place Market**: This is a historic public market in Seattle, where you

Now, let's reuse same session ID, change the model, and generate a new file with the new model.

In [12]:
print(f"Session ID: {session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "What are three fun things to do in Seattle on a rainy day?"
                    "Save your answer to a Markdown file with Sonnet prefix."
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "apac.amazon.nova-pro-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

Session ID: 1F7EFC37-77BE-48FE-B42B-D48B08946F67

<thinking>The user has requested the same information again but with a Sonnet prefix. I can reuse the previously created list of fun activities and add a Sonnet prefix to the Markdown file.</thinking>

Here are three fun things to do in Seattle on a rainy day:

1. **Visit the Museum of Pop Culture (MoPOP)**: This museum is dedicated to contemporary popular culture, including music, movies, and video games. It's a great place to spend a rainy day.

2. **Explore Pike Place Market**: This is a historic public market in Seattle, where you can find fresh produce, artisanal goods, and local crafts. It's a fun place to explore, even on a rainy day.

3. **Take a Tour of the Seattle Center**: This cultural and entertainment complex includes the Space Needle, the Museum of History & Industry, and the Experience Music Project. You can take a guided tour to learn about the history and culture of Seattle. 
[Tool: file_operations]


<thinking>The fil

### 6. Run Commands on the Agent's VM

Every Harness session runs in an isolated microVM. Use `execute_command` to run shell commands directly on the VM — useful for inspecting the environment or reading files the agent created.

> ⚠️ **Why not `!ls`?** Jupyter's `!` runs commands on your **local environment** where Jupyter is running. `execute_command` runs on the **agent's remote microVM** in AWS.

In [13]:
def run(cmd: str):
    print(f"$ {cmd}")
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=session_id,
        body={"command": cmd},
    )
    for event in resp["stream"]:
        if "chunk" in event:
            chunk = event["chunk"]
            if "contentDelta" in chunk:
                d = chunk["contentDelta"]
                if "stdout" in d:
                    print(d["stdout"], end="", flush=True)
                if "stderr" in d:
                    print(d["stderr"], end="", flush=True)
            elif "contentStop" in chunk:
                print(f"\n[exit: {chunk['contentStop']['exitCode']}]")
    print()

In [14]:
run("ls -la")

$ ls -la
total 16
drwxr-xr-x 1 root root 4096 Nov  2 13:20 .
drwxr-xr-x 1 root root 4096 Nov  2 13:02 ..
-rw-r--r-- 1 root root  703 Nov  2 13:17 fun_things_to_do_in_seattle.md
-rw-r--r-- 1 root root  715 Nov  2 13:20 sonnet_fun_things_to_do_in_seattle.md

[exit: 0]



In [15]:
run("pwd")

$ pwd
/home

[exit: 0]



In [16]:
# List files, then read each .md file
run('for f in *.md; do echo "=== $f ==="; cat "$f"; echo; done')

$ for f in *.md; do echo "=== $f ==="; cat "$f"; echo; done
=== fun_things_to_do_in_seattle.md ===
Here are three fun things to do in Seattle on a rainy day:

1. **Visit the Museum of Pop Culture (MoPOP)**: This museum is dedicated to contemporary popular culture, including music, movies, and video games. It's a great place to spend a rainy day.

2. **Explore Pike Place Market**: This is a historic public market in Seattle, where you can find fresh produce, artisanal goods, and local crafts. It's a fun place to explore, even on a rainy day.

3. **Take a Tour of the Seattle Center**: This cultural and entertainment complex includes the Space Needle, the Museum of History & Industry, and the Experience Music Project. You can take a guided tour to learn about the history and culture of Seattle.
=== sonnet_fun_things_to_do_in_seattle.md ===
**Sonnet**

Here are three fun things to do in Seattle on a rainy day:

1. **Visit the Museum of Pop Culture (MoPOP)**: This museum is dedicated to con

### 7. Cleanup

Delete the Harness agent and IAM role when done.

In [17]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

Deleted harness: GettingStarted_c94323ab-2mJIjUS4EN


In [18]:
delete_harness_role()

Deleted inline policy: HarnessExecutionPolicy
Deleted role: HarnessExecutionRole
